In [4]:
from xgboost import XGBClassifier

In [5]:
import pandas as pd

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv")["interaction_type"]
y_test = pd.read_csv("../data/processed/y_test.csv")["interaction_type"]

In [6]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(178108, 2)
(44527, 2)
(178108,)
(44527,)


In [8]:
import time
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate_model(model, X_train, X_test, y_train, y_test):
    start_time = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    results = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "Training Time": training_time
    }

    return results, y_pred

In [9]:
xgb_model = XGBClassifier(
    objective="multi:softmax",
    num_class=len(set(y_train)),
    n_estimators=100,
    learning_rate=0.1,
    max_depth=8,
    random_state=42,
    eval_metric="mlogloss"
)

xgb_results, xgb_pred = evaluate_model(
    xgb_model,
    X_train,
    X_test,
    y_train,
    y_test
)

xgb_results

{'Accuracy': 0.46019269207447167,
 'Macro Precision': 0.16438584893014965,
 'Macro Recall': 0.07896685796691356,
 'Macro F1': 0.09446452816507381,
 'Weighted F1': 0.4116465375723048,
 'Training Time': 22.85499596595764}

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import joblib

In [10]:
param_grid = {
    "n_estimators": [100],
    "max_depth": [20, None],
    "min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        class_weight="balanced",
        n_jobs=1
    ),
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=2,
    n_jobs=1,
    verbose=2
)

grid_search.fit(X_train, y_train)

Fitting 2 folds for each of 4 candidates, totalling 8 fits
[CV] END max_depth=20, min_samples_split=2, n_estimators=100; total time=   9.6s
[CV] END max_depth=20, min_samples_split=2, n_estimators=100; total time=   9.6s
[CV] END max_depth=20, min_samples_split=5, n_estimators=100; total time=   9.2s
[CV] END max_depth=20, min_samples_split=5, n_estimators=100; total time=   9.3s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=  17.8s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=  17.4s
[CV] END max_depth=None, min_samples_split=5, n_estimators=100; total time=  14.0s
[CV] END max_depth=None, min_samples_split=5, n_estimators=100; total time=  13.9s


GridSearchCV(cv=2,
             estimator=RandomForestClassifier(class_weight='balanced', n_jobs=1,
                                              random_state=42),
             n_jobs=1,
             param_grid={'max_depth': [20, None], 'min_samples_split': [2, 5],
                         'n_estimators': [100]},
             scoring='f1_weighted', verbose=2)

In [11]:
print(grid_search.best_params_)
print(grid_search.best_score_)

{'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
0.47639383325097784


In [12]:
best_model = grid_search.best_estimator_

import joblib
joblib.dump(best_model, "../models/best_model.pkl")

['../models/best_model.pkl']

In [13]:
from sklearn.metrics import accuracy_score, f1_score

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro", zero_division=0))
print("Weighted F1:", f1_score(y_test, y_pred, average="weighted", zero_division=0))

Accuracy: 0.5109708716059919
Macro F1: 0.2765205416974989
Weighted F1: 0.5205635251264557
